# Chapter 6: Practical High-Performance Kernels with Triton
### Companion notebook (High-Performance Machine Learning)

This notebook turns the chapter's claims into runnable, measurable code. It is written for a
**Colab T4** (Runtime → Change runtime type → T4 GPU), the same Turing-class card used in the
running RAG case study, but every cell runs on any recent NVIDIA GPU with a matching Triton build.

What you will build and measure here:

1. **Vector add**: your first kernel — the launch grid, the boundary mask, and a correctness check.
2. **Fused softmax**: one read and one write instead of many, benchmarked honestly against a naive
   multi-pass baseline **and** against `torch.nn.functional.softmax`.
3. **Fused LayerNorm / RMSNorm**: the normalization fusion that libraries like Liger-Kernel ship.
4. **Tiled matmul**: `tl.dot`, an FP32 accumulator, grouped ordering for L2 reuse, and the honest
   fraction of cuBLAS you actually reach.
5. **Autotuning**: searching block sizes, `num_warps`, and `num_stages`, and why it belongs in warmup.
6. **Debugging**: `TRITON_INTERPRET=1` and validation against a PyTorch reference at the right tolerance.

Every kernel is checked against a plain PyTorch reference with `torch.allclose`. A fast kernel that
computes the wrong answer is worse than the slow one it replaced (Principle 18).

> Solutions and extended versions of the exercises live at
> https://github.com/PacktPublishing/High-Performance-Machine-Learning

## 0. Setup

The chapter pins **PyTorch 2.5 + Triton 3.1** for reproducibility. Colab may ship a different pair;
the kernels here use only stable API and run across nearby versions, but if a cell fails on an API
name, check your Triton version first.

In [ ]:
import torch
import numpy as np

import triton
import triton.language as tl

assert torch.cuda.is_available(), 'Enable a GPU runtime (Colab: Runtime > Change runtime type > T4).'
dev = torch.device('cuda')
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f'GPU: {name}  (compute capability {cap[0]}.{cap[1]})')
print(f'PyTorch: {torch.__version__}   Triton: {triton.__version__}')

IS_TURING = (cap == (7, 5))
if IS_TURING:
    print('=> Turing-class card (T4): FP16 tensor cores, no BF16. Matches the case study.')

In [ ]:
def cuda_time(fn, iters=50, warmup=10):
    """Median ms per call, measured with CUDA events. Warm the kernel before timing (Chapter 5)."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
        s.record(); fn(); e.record(); torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    return float(np.median(times))

def gbps(bytes_moved, ms):
    """Effective memory bandwidth in GB/s for a kernel that moved `bytes_moved` bytes."""
    return bytes_moved / (ms * 1e-3) / 1e9

## 1. Your first kernel: vector add

One program instance handles one `BLOCK_SIZE`-element tile. `tl.program_id(0)` tells the instance
which tile it owns; `offsets` are its element indices; `mask = offsets < n_elements` guards the load
and the store so the final, partly-empty tile never reads or writes out of bounds (Figure 6.2).

In [ ]:
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, x + y, mask=mask)

def add(x, y, BLOCK_SIZE=1024):
    out = torch.empty_like(x)
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    add_kernel[grid](x, y, out, n_elements, BLOCK_SIZE=BLOCK_SIZE)
    return out

# Deliberately NOT a multiple of BLOCK_SIZE, to exercise the mask.
n = 1_000_003
x = torch.rand(n, device=dev)
y = torch.rand(n, device=dev)
out = add(x, y)

# --- self-check: matches PyTorch exactly for a pure elementwise add ---
assert torch.allclose(out, x + y, atol=1e-6), 'vector add does not match x + y'
ms = cuda_time(lambda: add(x, y))
bytes_moved = 3 * x.numel() * x.element_size()  # read x, read y, write out
print(f'vector add: {ms:.4f} ms   effective bandwidth ~ {gbps(bytes_moved, ms):.0f} GB/s')
print('self-check passed: Triton vector add matches x + y on a non-multiple-of-block length.')

## 2. Fusion as the core lever: fused softmax

Softmax is memory-bound: it does little arithmetic per element, so its time is dominated by moving
the array through DRAM. A naive implementation walks the array several times (max, subtract, exp,
sum, divide), paying a DRAM round-trip each pass (Figure 6.4). The fused kernel loads each row once,
does every step on chip, and writes once.

Two details matter for correctness. The masked load uses `other=-float('inf')` so padding lanes lose
the `max` and contribute zero to the `sum` after `exp`; using `0.0` here would corrupt both reductions.
And the row is cast to FP32 on chip for numerical stability.

In [ ]:
@triton.jit
def softmax_kernel(out_ptr, in_ptr, in_row_stride, out_row_stride,
                   n_rows, n_cols, BLOCK_SIZE: tl.constexpr):
    # Persistent, grid-stride loop over rows: one program handles many rows.
    row_start = tl.program_id(0)
    num_programs = tl.num_programs(0)
    for row in range(row_start, n_rows, num_programs):
        col_offsets = tl.arange(0, BLOCK_SIZE)
        mask = col_offsets < n_cols
        row_ptr = in_ptr + row * in_row_stride + col_offsets
        # -inf for out-of-range lanes: they must lose the max and vanish under exp.
        x = tl.load(row_ptr, mask=mask, other=-float('inf'))
        x = x.to(tl.float32)
        x = x - tl.max(x, axis=0)             # subtract max for stability
        num = tl.exp(x)
        denom = tl.sum(num, axis=0)
        y = num / denom
        out_row_ptr = out_ptr + row * out_row_stride + col_offsets
        tl.store(out_row_ptr, y, mask=mask)

def softmax(x):
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)   # heuristic: one tile covers a row
    out = torch.empty_like(x)
    # A modest number of persistent programs; each sweeps many rows.
    grid = (min(n_rows, 2048),)
    softmax_kernel[grid](out, x, x.stride(0), out.stride(0),
                         n_rows, n_cols, BLOCK_SIZE=BLOCK_SIZE)
    return out

x = torch.randn(4096, 2048, device=dev)
y_triton = softmax(x)
y_ref = torch.softmax(x, axis=1)

# --- self-check: matches torch.softmax ---
assert torch.allclose(y_triton, y_ref, atol=1e-5), 'fused softmax does not match torch.softmax'
print('self-check passed: fused softmax matches torch.softmax within 1e-5.')

### The honest benchmark

We compare three implementations: a naive multi-pass PyTorch softmax (many DRAM round-trips), the
fused Triton kernel, and `torch.nn.functional.softmax` (already a fused CUDA kernel). Expect a large
win over the naive baseline and a **small or negligible** win over `F.softmax` — the library is
already fused, so the fusion argument no longer separates them. Reporting both gaps is the point.

In [ ]:
import torch.nn.functional as F

def naive_softmax(x):
    # Each line is a separate CUDA kernel => a separate DRAM round-trip.
    x_max = x.max(dim=1, keepdim=True)[0]
    z = x - x_max
    num = torch.exp(z)
    denom = num.sum(dim=1, keepdim=True)
    return num / denom

ms_naive = cuda_time(lambda: naive_softmax(x))
ms_triton = cuda_time(lambda: softmax(x))
ms_torch = cuda_time(lambda: F.softmax(x, dim=1))

print(f'naive multi-pass : {ms_naive:.4f} ms')
print(f'fused Triton     : {ms_triton:.4f} ms   ({ms_naive/ms_triton:.2f}x vs naive)')
print(f'F.softmax        : {ms_torch:.4f} ms   ({ms_torch/ms_triton:.2f}x = Triton over torch)')
print('\nInterpretation: the big win is over the NAIVE baseline, because fusion removes its extra')
print('DRAM round-trips. Against F.softmax the gap is small: it is already fused (Principle 15).')

# --- self-check: fusion beats the naive multi-pass baseline ---
assert ms_triton < ms_naive, f'expected fused < naive, got fused={ms_triton:.4f} naive={ms_naive:.4f}'
print('self-check passed: the fused kernel beats the naive multi-pass softmax.')

## 3. Fusing a normalization layer: LayerNorm and RMSNorm

Normalization is the fusion target that pays off on real training and inference workloads (this is
where Liger-Kernel and Unsloth spend their Triton effort). The kernel reads a row, computes mean and
variance, normalizes, and applies the affine transform — all on chip. The row is read up to three
times, but from registers/SRAM, not DRAM; from DRAM it is one read and one write. Reductions run in
FP32 regardless of storage dtype.

In [ ]:
@triton.jit
def layernorm_fwd_kernel(x_ptr, w_ptr, b_ptr, out_ptr, x_row_stride, out_row_stride,
                         n_cols, eps, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    x = tl.load(x_ptr + row * x_row_stride + cols, mask=mask, other=0.0).to(tl.float32)
    mean = tl.sum(x, axis=0) / n_cols
    xc = tl.where(mask, x - mean, 0.0)          # zero the padding before squaring
    var = tl.sum(xc * xc, axis=0) / n_cols
    rstd = 1.0 / tl.sqrt(var + eps)
    w = tl.load(w_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    b = tl.load(b_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    y = xc * rstd * w + b
    tl.store(out_ptr + row * out_row_stride + cols, y, mask=mask)

def layernorm(x, weight, bias, eps=1e-5):
    n_rows, n_cols = x.shape
    out = torch.empty_like(x)
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    layernorm_fwd_kernel[(n_rows,)](x, weight, bias, out, x.stride(0), out.stride(0),
                                    n_cols, eps, BLOCK_SIZE=BLOCK_SIZE)
    return out

# Use a non-power-of-two feature dim on purpose (exercises the mask).
rows, feat = 4096, 1536
x = torch.randn(rows, feat, device=dev)
w = torch.randn(feat, device=dev)
b = torch.randn(feat, device=dev)
y_triton = layernorm(x, w, b)
y_ref = F.layer_norm(x, (feat,), w, b, eps=1e-5)

# --- self-check: matches F.layer_norm ---
assert torch.allclose(y_triton, y_ref, atol=1e-4), 'fused LayerNorm does not match F.layer_norm'
print('self-check passed: fused LayerNorm matches F.layer_norm within 1e-4 on a non-power-of-two dim.')

In [ ]:
# RMSNorm: drop the mean pass and the bias, normalize by the root-mean-square.
# (Exercise 6.3 asks you to write this from scratch; here is a reference to check against.)
@triton.jit
def rmsnorm_fwd_kernel(x_ptr, w_ptr, out_ptr, x_row_stride, out_row_stride,
                       n_cols, eps, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    x = tl.load(x_ptr + row * x_row_stride + cols, mask=mask, other=0.0).to(tl.float32)
    ms = tl.sum(x * x, axis=0) / n_cols          # mean square; padding is 0 so it does not contribute
    rstd = 1.0 / tl.sqrt(ms + eps)
    w = tl.load(w_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    y = x * rstd * w
    tl.store(out_ptr + row * out_row_stride + cols, y, mask=mask)

def rmsnorm(x, weight, eps=1e-6):
    n_rows, n_cols = x.shape
    out = torch.empty_like(x)
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    rmsnorm_fwd_kernel[(n_rows,)](x, weight, out, x.stride(0), out.stride(0),
                                  n_cols, eps, BLOCK_SIZE=BLOCK_SIZE)
    return out

def rmsnorm_ref(x, weight, eps=1e-6):
    xf = x.float()
    rms = torch.rsqrt(xf.pow(2).mean(dim=1, keepdim=True) + eps)
    return (xf * rms * weight).to(x.dtype)

y_triton = rmsnorm(x, w)
y_ref = rmsnorm_ref(x, w)
assert torch.allclose(y_triton, y_ref, atol=1e-4), 'RMSNorm does not match the reference'
print('self-check passed: fused RMSNorm matches the PyTorch reference within 1e-4.')

## 4. Tiled matrix multiplication

The compute-bound counterpart. Each program computes one `BLOCK_M × BLOCK_N` output tile, walking the
K dimension and accumulating with `tl.dot` into an **FP32** accumulator (narrow storage, wide
accumulation — exactly what cuBLAS does). The `pid` de-linearization implements grouped ordering so a
small working set of A and B tiles stays hot in L2 (Figure 6.7).

The honest lesson: a careful Triton GEMM reaches most of the way to cuBLAS and rarely beats it
(Principle 16).

In [ ]:
@triton.jit
def matmul_kernel(a_ptr, b_ptr, c_ptr, M, N, K,
                  stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                  BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr,
                  BLOCK_SIZE_K: tl.constexpr, GROUP_SIZE_M: tl.constexpr):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    # Grouped ordering: keep A and B tiles hot in L2 across nearby output tiles.
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        accumulator = tl.dot(a, b, accumulator)
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    c = accumulator.to(tl.float16)

    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

def matmul(a, b, BM=64, BN=64, BK=32, GROUP_M=8):
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, 'inner dimensions must match'
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = lambda meta: (triton.cdiv(M, meta['BLOCK_SIZE_M']) * triton.cdiv(N, meta['BLOCK_SIZE_N']),)
    matmul_kernel[grid](a, b, c, M, N, K,
                        a.stride(0), a.stride(1), b.stride(0), b.stride(1), c.stride(0), c.stride(1),
                        BLOCK_SIZE_M=BM, BLOCK_SIZE_N=BN, BLOCK_SIZE_K=BK, GROUP_SIZE_M=GROUP_M)
    return c

M = N = K = 1024
a = torch.randn(M, K, device=dev, dtype=torch.float16)
b = torch.randn(K, N, device=dev, dtype=torch.float16)
c_triton = matmul(a, b)
c_ref = torch.matmul(a, b)

# --- self-check: FP16 tolerance, because FP32-accumulated tl.dot differs from eager in the low bits ---
assert torch.allclose(c_triton, c_ref, atol=1e-1, rtol=1e-2), 'matmul does not match torch.matmul'
print('self-check passed: Triton matmul matches torch.matmul at FP16 tolerance.')

In [ ]:
# How close to cuBLAS? Report the fraction, not a claim of victory.
ms_triton = cuda_time(lambda: matmul(a, b))
ms_cublas = cuda_time(lambda: torch.matmul(a, b))
flops = 2 * M * N * K
print(f'Triton matmul : {ms_triton:.4f} ms   ({flops/(ms_triton*1e-3)/1e12:.1f} TFLOP/s)')
print(f'cuBLAS        : {ms_cublas:.4f} ms   ({flops/(ms_cublas*1e-3)/1e12:.1f} TFLOP/s)')
print(f'fraction of cuBLAS reached: {ms_cublas/ms_triton*100:.0f}%')
print('\nExpect well below 100% with these small hand-picked block sizes. Autotuning (next section)')
print('closes much of the gap; beating cuBLAS is not the goal (Principle 16).')

In [ ]:
# The effect of grouped ordering: disable it (GROUP_M=1) and compare.
ms_grouped = cuda_time(lambda: matmul(a, b, GROUP_M=8))
ms_rowmajor = cuda_time(lambda: matmul(a, b, GROUP_M=1))
print(f'GROUP_SIZE_M=8 (grouped) : {ms_grouped:.4f} ms')
print(f'GROUP_SIZE_M=1 (row-major): {ms_rowmajor:.4f} ms')
print('Grouped ordering changes no arithmetic; any speedup is pure L2 reuse (Figure 6.7).')

## 5. Autotuning

The best block sizes, `num_warps`, and `num_stages` are not derivable — they must be searched.
`triton.autotune` benchmarks each candidate the first time it sees a new `(M, N, K)` shape and caches
the winner. The first call therefore pays for the whole search, which is why autotuning belongs in
**warmup**, not on the latency-critical path.

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 8}, num_stages=3, num_warps=8),
        triton.Config({'BLOCK_SIZE_M': 64,  'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64,  'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64,  'BLOCK_SIZE_N': 64,  'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4, num_warps=4),
    ],
    key=['M', 'N', 'K'],
)
@triton.jit
def matmul_kernel_tuned(a_ptr, b_ptr, c_ptr, M, N, K,
                        stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr,
                        BLOCK_SIZE_K: tl.constexpr, GROUP_SIZE_M: tl.constexpr):
    # Body identical to matmul_kernel above.
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        accumulator = tl.dot(a, b, accumulator)
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    c = accumulator.to(tl.float16)
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

def matmul_tuned(a, b):
    M, K = a.shape; _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = lambda meta: (triton.cdiv(M, meta['BLOCK_SIZE_M']) * triton.cdiv(N, meta['BLOCK_SIZE_N']),)
    matmul_kernel_tuned[grid](a, b, c, M, N, K,
                              a.stride(0), a.stride(1), b.stride(0), b.stride(1), c.stride(0), c.stride(1))
    return c

import time
M = N = K = 2048
a = torch.randn(M, K, device=dev, dtype=torch.float16)
b = torch.randn(K, N, device=dev, dtype=torch.float16)

# First call: pays for the autotuning search over all configs.
torch.cuda.synchronize(); t0 = time.perf_counter()
_ = matmul_tuned(a, b); torch.cuda.synchronize()
first_call_s = time.perf_counter() - t0

# Steady state: winning config is cached for this shape.
ms_steady = cuda_time(lambda: matmul_tuned(a, b))
print(f'first call (incl. search): {first_call_s*1e3:.1f} ms')
print(f'steady-state latency     : {ms_steady:.4f} ms')
print('The search cost is paid once per shape, at warmup. Never autotune on the hot path.')

# --- self-check: tuned kernel is still correct ---
assert torch.allclose(matmul_tuned(a, b), torch.matmul(a, b), atol=1e-1, rtol=1e-2)
print('self-check passed: autotuned matmul matches torch.matmul at FP16 tolerance.')

## 6. Debugging with the interpreter

`TRITON_INTERPRET=1` runs kernels on the CPU in pure Python, one program instance at a time, so you
can `print` tile values and step through pointer arithmetic and masks. It is slow and for debugging
only, but it catches the index and mask errors that are otherwise nearly invisible on the GPU.

The variable must be set **before** Triton is imported, so in a notebook you set it and restart, or
run the kernel in a fresh subprocess. The cell below writes a tiny script and runs it with the flag
set, so you can see the interpreter without disturbing this session.

In [ ]:
script = r'''
import os
os.environ["TRITON_INTERPRET"] = "1"   # must be set before importing triton
import torch, triton, triton.language as tl

@triton.jit
def dbg_add(x_ptr, y_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < n
    if pid == 0:
        print("program 0 offsets:", offs)
        print("program 0 mask   :", mask)
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)

n = 6
x = torch.arange(n, dtype=torch.float32)
y = torch.ones(n)
out = torch.empty(n)
dbg_add[(triton.cdiv(n, 4),)](x, y, out, n, BLOCK_SIZE=4)
print("out:", out)
'''
with open('dbg_kernel.py', 'w') as f:
    f.write(script)

import subprocess, sys
res = subprocess.run([sys.executable, 'dbg_kernel.py'], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print('stderr:', res.stderr)
print('Notice you could print tile values directly — impossible on a compiled GPU kernel.')

## 7. Exercises

These match the Hands-On Exercises in the chapter. Reference solutions are in the repo.

**6.1 — Vector add and the launch grid.** Sweep `BLOCK_SIZE` over 128, 256, 512, 1024, 2048 and plot
effective bandwidth. Explain why the kernel is memory-bound and why larger blocks stop helping past a
point. Validate against `x + y`.

**6.2 — Fused softmax versus baselines.** Reproduce the three-way benchmark (naive, Triton,
`F.softmax`) and explain each gap with the memory-traffic argument. Validate on a row that is all
negative, and explain why that row tests the `other` value.

**6.3 — Write RMSNorm from the LayerNorm template.** Starting from `layernorm_fwd_kernel`, write an
RMSNorm kernel yourself (remove the mean pass and bias). Validate against `rmsnorm_ref` at FP16
tolerance on a non-power-of-two feature dim. Compare to the provided `rmsnorm_fwd_kernel`.

**6.4 — Matmul against cuBLAS.** Compare `matmul` to `torch.matmul` across sizes 512, 1024, 2048,
4096. Report the fraction of cuBLAS reached. Then set `GROUP_M=1` and measure the L2-reuse penalty.

**6.5 — Autotuning.** Extend the config list, measure first-call cost versus steady-state latency
separately, and report the winning config for your GPU and shape.

**6.6 — The interpreter finds a bug.** Introduce an off-by-one in a mask (e.g. `offs <= n`). Observe
the GPU symptom, then rerun under `TRITON_INTERPRET=1` and use `print` to locate the bad lane.

**6.7 — Build-or-buy decision.** For three RAG-pipeline operations, classify each as compute- or
memory-bound, estimate its Amdahl share, and decide library / `torch.compile` / custom kernel.

**6.8 — Register a custom op (stretch).** Wrap `rmsnorm_fwd_kernel` as a `torch.library.triton_op`
with a backward. Confirm a gradient check against the PyTorch reference and no graph break under
`torch.compile`.